In [1]:
# !pip install chromadb
# %pip install sentence-transformers
# %pip install pypdf

In [2]:
# IMPORT LIBRARIES

import os
import chromadb
import autogen
import sentence_transformers
import pypdf

from pathlib import Path

from autogen import AssistantAgent
from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent

from autogen.coding import LocalCommandLineCodeExecutor


C:\Users\Raji\anaconda3\envs\ai_agents\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# OLLAMA MODEL CONFIGURATION

config_list = [{"model": "llama3.2", "api_type": "ollama", 
                "client_host": "http://localhost:11434", "stream": False}]

print("Model selected:", config_list[0]["model"])

Model selected: llama3.2


In [4]:
# CHECK PDF FILE

pdf_file = Path("book.pdf")

if not pdf_file.exists():
    raise "FileNotFoundError"("book.pdf not found. Place the PDF in the notebook folder.")

print("PDF file found:", pdf_file.resolve())

PDF file found: C:\Hope AI\AI Course Tamil\16. AI Agents\book.pdf


<>:6: SyntaxWarning: 'str' object is not callable; perhaps you missed a comma?
<>:6: SyntaxWarning: 'str' object is not callable; perhaps you missed a comma?
C:\Users\Raji\AppData\Local\Temp\ipykernel_14500\3436133074.py:6: SyntaxWarning: 'str' object is not callable; perhaps you missed a comma?
  raise "FileNotFoundError"("book.pdf not found. Place the PDF in the notebook folder.")


In [5]:
# CREATE PYTHON CODE EXECUTOR

work_dir = Path("autogen_output")
work_dir.mkdir(exist_ok = True)

code_executor = LocalCommandLineCodeExecutor(work_dir=work_dir)

# CREATE ASSISTANT AGENT

assistant = AssistantAgent(name = "assistant", 
                           system_message = "You are a helpful assistant. Answer using the retrieved PDF context.",
                           llm_config = {"timeout": 600, "cache_seed": 42, "config_list": config_list})

# CREATE RETRIEVAL USER PROXY AGENT

ragproxyagent = RetrieveUserProxyAgent(
    name = "ragproxyagent", human_input_mode = "NEVER", max_consecutive_auto_reply = 3, 
    retrieve_config = {
        "task": "qa", "docs_path": ["book.pdf"],
        "chunk_token_size": 200, "model": config_list[0]["model"],
        "vector_db": "chroma", "collection_name": "book_collection",
        "overwrite": True, "get_or_create": True},
    code_execution_config=False)

print("Both agents created successfully!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1037.21it/s]


Both agents created successfully!


In [7]:
import pypdf
import autogen.retrieve_utils as retrieve_utils

# Get the actual PDF extraction function
pdf_function = retrieve_utils.extract_text_from_pdf

# Add pypdf to the function's global namespace
pdf_function.__globals__["pypdf"] = pypdf

# Also add it to the retrieve_utils module
retrieve_utils.pypdf = pypdf

print("PyPDF version:", pypdf.__version__)
print("PyPDF added to function namespace:",
      "pypdf" in pdf_function.__globals__)

print("Fix applied successfully!")

PyPDF version: 7.0.1
PyPDF added to function namespace: True
Fix applied successfully!


In [8]:
# FIRST TASK - SUMMARIZE THE BOOK

assistant.reset()

qa_problem = """
Summarize the book.

Requirements:

1. Explain the main topics.
2. Identify important concepts.
3. Explain everything in simple language.
4. Include examples.
5. Explain mathematical calculations wherever available.
"""

chat_result = ragproxyagent.initiate_chat(assistant, message = ragproxyagent.message_generator,
                                          problem = qa_problem)

Trying to create collection.


max_tokens is too small to fit a single line of text. Breaking this line:
	The 22 ...
Failed to split docs with must_break_at_empty_line being True, set to False.
2026-11-14 13:35:24,353 - autogen.agentchat.contrib.retrieve_user_proxy_agent - INFO - Found 215 chunks.
2026-11-14 13:35:24,360 - autogen.agentchat.contrib.vectordb.chromadb - INFO - No content embedding is provided. Will use the VectorDB's embedding function to generate the content embedding.
Model llama3.2 not found. Using cl100k_base encoding.


VectorDB returns doc_ids:  [['f0804281', '150c596a', '8bf7f8ee', '230f3fea', 'cc6f0740', 'e7d73411', '5b19ade7', '89fd1dac', '81d8712b', '8a2a101c', '451a2a58', '7e3514a1', 'd11d708f', '62e6169a', 'c1b21496', '05dac356', 'f5837698', '217a91d7', 'd0e85c7b', 'e75197c0']]
Adding content of doc f0804281 to context.


Model llama3.2 not found. Using cl100k_base encoding.


Adding content of doc 150c596a to context.


Model llama3.2 not found. Using cl100k_base encoding.


... (30 outputs left out)

Adding content of doc 217a91d7 to context.


Model llama3.2 not found. Using cl100k_base encoding.


ragproxyagent (to assistant):

You're a retrieve augmented chatbot. You answer user's questions based on your own knowledge and the
context provided by the user.
If you can't answer the question with or without the current context, you should reply exactly `UPDATE CONTEXT`.
You must give as short an answer as possible.

User's question is: 
Summarize the book.

Requirements:

1. Explain the main topics.
2. Identify important concepts.
3. Explain everything in simple language.
4. Include examples.
5. Explain mathematical calculations wherever available.


Context is: 123
THE LA W OF ACCELERATION
22 Laws of Marketing  10/31/02  12:23 PM  Page 12322
The Law of 
Resources
Without adequate funding
an idea won’t get off the ground.
124
22 Laws of Marketing  10/31/02  12:23 PM  Page 124If you have a good idea and you’ve picked up this
book with the thought in mind that all you need is a lit-
tle marketing help, this chapter will throw cold water
on that thought.
Even the best idea in the worl

assistant (to ragproxyagent):

Here's a summary of the first chapter of "The 22 Immutable Laws of Marketing" by Al Ries and Jack Trout:

**The Law of Leadership**

* The law states that leadership means being the best in the market.
* It emphasizes the importance of innovation and differentiation.
* Companies should focus on leading their category rather than trying to be a best-of-breed product.

Key points:

* Benchmarking doesn't work because people perceive the first product into their mind as superior.
* Marketing is a battle of perceptions, not products.
* Trying harder or being better is not the secret of marketing success.
* The law of leadership requires companies to innovate and differentiate themselves from others in the market.

**The Law of Perception**

* People tend to remember the first product they experience, even if it's not necessarily the best one.
* Marketing is about creating a perception that is positive and desirable.
* Companies should focus on creating an ima

In [10]:
# SECOND TASK - CREATIVE BRAINSTORMING

assistant.reset()

brainstorm_problem = """
You are a creative brainstorming expert.

Based on the knowledge available in the book:

1. Generate creative ideas.
2. Identify practical applications.
3. Suggest innovative solutions.
4. Explain how these ideas can be implemented.
5. Provide examples wherever possible.

Use information from the book as the foundation
for your brainstorming.
"""

chat_result = ragproxyagent.initiate_chat(assistant, message = ragproxyagent.message_generator,
                                          problem = brainstorm_problem)

VectorDB returns doc_ids:  [['f0804281', '0e08a976', '81d8712b', '21ec236c', '7e1ac0cd', '8bf7f8ee', '150c596a', '89fd1dac', 'be843ac3', '5b19ade7', '230f3fea', 'a5e9b671', 'ba9a27e8', 'd5a791b1', '4cf44936', 'f5837698', 'd11d708f', '8126c05a', '451a2a58', 'a1f246ff']]
Adding content of doc f0804281 to context.


Adding content of doc 0e08a976 to context.


Adding content of doc 81d8712b to context.


Adding content of doc 21ec236c to context.


Adding content of doc 7e1ac0cd to context.


Adding content of doc 8bf7f8ee to context.


... (9 outputs left out)

Adding content of doc f5837698 to context.


Adding content of doc d11d708f to context.


ragproxyagent (to assistant):

You're a retrieve augmented chatbot. You answer user's questions based on your own knowledge and the
context provided by the user.
If you can't answer the question with or without the current context, you should reply exactly `UPDATE CONTEXT`.
You must give as short an answer as possible.

User's question is: 
You are a creative brainstorming expert.

Based on the knowledge available in the book:

1. Generate creative ideas.
2. Identify practical applications.
3. Suggest innovative solutions.
4. Explain how these ideas can be implemented.
5. Provide examples wherever possible.

Use information from the book as the foundation
for your brainstorming.


Context is: 123
THE LA W OF ACCELERATION
22 Laws of Marketing  10/31/02  12:23 PM  Page 12322
The Law of 
Resources
Without adequate funding
an idea won’t get off the ground.
124
22 Laws of Marketing  10/31/02  12:23 PM  Page 124If you have a good idea and you’ve picked up this
book with the thought in mind t

assistant (to ragproxyagent):

Here are the notes from the chapter on "The Law of Unpredictability":

**Avis and Rent-a-Cars**

* Avis is only No. 2 in rent-a-cars, but it's not a bad idea to use this as a marketing concept
* The goal is to make Avis stand out and be seen as second best

**Positive Thinking**

* Positive thinking has been overrated in marketing
* Marketing efforts should focus on using ideas and concepts already installed in the brain
* Admitting problems can help companies connect with their audience

**Gourmet Popping Corn**

* Orville Redenbacher's Gourmet Popping Corn took a risk by charging twice as much for high-end popcorn
* The risk paid off, showing that people are willing to pay more for quality products

**No Predicting the Future**

* No one can predict the future with certainty
* Marketing plans should not try to predict the future, but instead focus on using ideas and concepts already installed in the brain

**Success and Arrogance**

* Success often lead

In [11]:
# THIRD TASK - MARKETING IDEAS

assistant.reset()

ragproxyagent.human_input_mode = "ALWAYS"

marketing_problem = """
You are a marketing strategist.

Based on the knowledge available in the book,
suggest 5 creative marketing ideas for an
AI and Data Science training institute.

For each idea, provide:

1. Marketing idea name.
2. Description.
3. Target audience.
4. Implementation strategy.
5. Expected benefits.
6. Tools and technologies required.

If any coding is required:

- Use Python.
- Generate complete executable code.
- Save the Python script as a .py file
  in the autogen_output directory.

Make the marketing ideas practical,
creative and suitable for a training institute.
"""

chat_result = ragproxyagent.initiate_chat(assistant, message = ragproxyagent.message_generator,
                                          problem = marketing_problem)

print("\nAll AutoGen tasks completed!")

VectorDB returns doc_ids:  [['f0804281', '89fd1dac', '0e08a976', '150c596a', '81d8712b', 'ba9a27e8', '8bf7f8ee', '4cf44936', 'be843ac3', '8a2a101c', 'b210f301', '788184ee', 'd5a791b1', 'f5837698', '451a2a58', '4e3d050e', '083871f5', 'ff30ed4d', 'a1f246ff', '08e00728']]
Adding content of doc f0804281 to context.


Adding content of doc 89fd1dac to context.


Adding content of doc 0e08a976 to context.


Adding content of doc 150c596a to context.


Adding content of doc 81d8712b to context.


Adding content of doc ba9a27e8 to context.


... (11 outputs left out)

ragproxyagent (to assistant):

You're a retrieve augmented chatbot. You answer user's questions based on your own knowledge and the
context provided by the user.
If you can't answer the question with or without the current context, you should reply exactly `UPDATE CONTEXT`.
You must give as short an answer as possible.

User's question is: 
You are a marketing strategist.

Based on the knowledge available in the book,
suggest 5 creative marketing ideas for an
AI and Data Science training institute.

For each idea, provide:

1. Marketing idea name.
2. Description.
3. Target audience.
4. Implementation strategy.
5. Expected benefits.
6. Tools and technologies required.

If any coding is required:

- Use Python.
- Generate complete executable code.
- Save the Python script as a .py file
  in the autogen_output directory.

Make the marketing ideas practical,
creative and suitable for a training institute.


Context is: 123
THE LA W OF ACCELERATION
22 Laws of Marketing  10/31/02  12:23 PM  

assistant (to ragproxyagent):

The passage highlights 3 laws of marketing:

1. **The Law of Unpredictability**: No one can predict the future with certainty. Marketing plans should not try to do so. Instead, they should be prepared for anything and take calculated risks.
2. **The Law of Success**: Success often leads to arrogance and ultimately failure. Ego can hinder successful marketing by causing leaders in a category to underestimate their competitors and overestimate their own abilities.
3. **The Law of the Ladder**: The strategy used depends on which rung you occupy on the ladder. Companies should understand that prospects use a hierarchy in making decisions, with each product having its own "rung" in the mind. Marketing strategies should reflect this understanding, taking into account which position to aim for (first, second, or third) and how to acknowledge and utilize it.

These laws emphasize the importance of flexibility, humility, and adaptability in marketing, as well as u

Replying as ragproxyagent. Provide feedback to assistant. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  



>>>>>>>> NO HUMAN INPUT RECEIVED.

>>>>>>>> TERMINATING RUN (1afbb501-3e86-4211-90f9-0f9d74daef75): Termination message condition on agent 'ragproxyagent' met

All AutoGen tasks completed!
